# Myllia: Direction Notebook (Bilinear Conditional Factorization)

This notebook implements a medium-large architecture shift:

- Learn a low-rank interaction between perturbed gene embeddings and output gene embeddings
- Predict the full delta vector as a bilinear form with rank `R`
- Train with a metric-aligned weighted L1 proxy and evaluate with the official `myllia_score`

Core model:
\[
\hat D_{i,j} = \langle W_p z_{g_i},\; W_o u_j \rangle + b_j + b_i
\]
where:
- `z_{g_i}` is an embedding for the perturbed gene `g_i`
- `u_j` is an embedding for output gene `j`
- `R` is a small rank (16 to 64)

This uses `training_cells.h5ad` to build output gene embeddings.


In [1]:
# -----------------------------
# Imports and global settings
# -----------------------------
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

from myllia_metric import myllia_score

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

# Model / training hyperparams (start simple, adjust later)
EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT  = 128   # output gene embedding dim (from SVD)
RANK_R       = 32    # low-rank interaction size

DROPOUT      = 0.10
LR           = 2e-3
WD           = 1e-4
EPOCHS       = 1000
BATCH_GENES  = 16      # minibatch over perturbed genes
EVAL_EVERY   = 25
PATIENCE     = 12      # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")


In [2]:
# -----------------------------
# Scoring utilities (official metric)
# -----------------------------
def score_delta(dt: np.ndarray, dp: np.ndarray) -> dict:
    dt = dt.astype(np.float32, copy=False)
    dp = dp.astype(np.float32, copy=False)
    r = myllia_score(dt, dp)
    return {
        "score": float(r.score),
        "wcos": float(r.wcos),
        "mean_term": float(r.mean_term),
        "pred_wmae": float(r.pred_wmae),
    }


In [3]:
# -----------------------------
# Load means + mappings, build training deltas
# -----------------------------
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :]   # (80, 5127) delta vs non-targeting

delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [4]:
# -----------------------------
# Build gene embeddings from training_cells.h5ad (SVD on normalized expression)
# We build embeddings for a union set:
#   - output genes (5127)
#   - training pert genes (80)
#   - val pert genes (60)
# -----------------------------
def find_h5ad():
    candidates = [
        ROOT / "data" / "training_cells.h5ad",
        ROOT / "Data" / "training_cells.h5ad",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("training_cells.h5ad not found in data/ or Data/")

h5ad_path = find_h5ad()
adata = ad.read_h5ad(str(h5ad_path))

val_targets = df_valmap["pert"].astype(str).tolist()
union_genes = sorted(set([g.upper() for g in gene_columns] +
                         [g.upper() for g in train_genes.tolist()] +
                         [g.upper() for g in val_targets]))

# Normalize on ALL genes first, then subset to union genes
adata_u = adata.copy()
sc.pp.normalize_total(adata_u, target_sum=1e4, inplace=True)

varU = pd.Index([str(v).upper() for v in adata_u.var_names])
pos = varU.get_indexer(union_genes)
ok = pos >= 0
union_ok = [union_genes[i] for i in range(len(union_genes)) if ok[i]]
pos_ok = pos[ok]

missing = [union_genes[i] for i in range(len(union_genes)) if not ok[i]]
if missing:
    print(f"[warn] {len(missing)} / {len(union_genes)} union genes missing from h5ad. Example: {missing[:12]}")

adata_u = adata_u[:, pos_ok].copy()

X = adata_u.X
if not sparse.issparse(X):
    X = sparse.csr_matrix(X)
else:
    X = X.tocsr(copy=True)

# log2(1 + x)
X.data = np.log2(X.data + 1.0).astype(np.float32)

print("Embedding matrix X:", X.shape, "nnz:", X.nnz)

# SVD gives components_ (k, n_genes), transpose to get gene embeddings (n_genes, k)
svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (n_union_ok, k)
k_svd = gene_emb_all.shape[1]
print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)

gene2emb_pert = {union_ok[i]: gene_emb_all[i, :EMB_DIM_PERT].copy() for i in range(len(union_ok))}
gene2emb_out  = {union_ok[i]: gene_emb_all[i, :EMB_DIM_OUT ].copy() for i in range(len(union_ok))}

emb_fallback_pert = gene_emb_all[:, :EMB_DIM_PERT].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :EMB_DIM_OUT ].mean(axis=0).astype(np.float32)

def emb_pert(g: str) -> np.ndarray:
    return gene2emb_pert.get(str(g).upper(), emb_fallback_pert)

def emb_out(g: str) -> np.ndarray:
    return gene2emb_out.get(str(g).upper(), emb_fallback_out)

# Output gene embeddings in the exact output gene order
U_out = np.vstack([emb_out(g) for g in gene_columns]).astype(np.float32)  # (G, d_out)
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)  # (N, d_pert)

print("U_out:", U_out.shape, "Z_train:", Z_train.shape)


Embedding matrix X: (17882, 5143) nnz: 37968190
SVD k: 128 gene_emb_all: (5143, 128)
U_out: (5127, 128) Z_train: (80, 128)


In [21]:
# -----------------------------
# Torch: metric-aligned proxy loss (weighted L1-like)
# -----------------------------
def gate_smoothstep(x: torch.Tensor, a: float = GATE_A, b: float = GATE_B) -> torch.Tensor:
    if b <= a:
        raise ValueError("gate_smoothstep requires b > a")
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)
    err = torch.abs(delta_pred - delta_true)
    num = torch.sum(w * err, dim=1)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)
    return torch.mean(num / den)


In [17]:
# -----------------------------
# Bilinear conditional factorization model
# -----------------------------
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert: int, d_out: int, rank_r: int, dropout: float):
        super().__init__()
        self.rank_r = rank_r

        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # biases
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None  # set via set_gene_bias

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert: torch.Tensor, u_out: torch.Tensor) -> torch.Tensor:
        p = self.proj_p(z_pert)    # (B, R)
        o = self.proj_o(u_out)     # (G, R)
        y = p @ o.T                # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y


In [22]:
# -----------------------------
# Prepare tensors
# -----------------------------
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device)          # (G, d_out)
Zt = torch.tensor(Z_train, device=device)          # (N, d_pert)
Yt = torch.tensor(Y, device=device)                # (N, G)

print("N:", N, "G:", G, "device:", device)


N: 80 G: 5127 device: cuda


In [23]:
# -----------------------------
# Honest gene-level CV training loop
# -----------------------------
def train_one_fold(tr_idx, va_idx):
    model = BilinearDeltaModel(d_pert=EMB_DIM_PERT, d_out=EMB_DIM_OUT, rank_r=RANK_R, dropout=DROPOUT).to(device)
    model.set_gene_bias(G)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            loss = weighted_l1_like(Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]
            s = score_delta(va_true, va_pred)
            sc = s["score"]

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_state

kf = KFold(n_splits=8, shuffle=True, random_state=SEED)
fold_scores = []
fold_states = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):
    best_score, best_state = train_one_fold(tr_idx, va_idx)
    fold_scores.append(float(best_score))
    fold_states.append(best_state)
    print(f"fold {fold}: best_score={best_score:.6f}")

print(f"[cv] mean={float(np.mean(fold_scores)):.6f} std={float(np.std(fold_scores)):.6f}")


fold 1: best_score=0.165036
fold 2: best_score=0.109817
fold 3: best_score=0.098343
fold 4: best_score=0.139945
fold 5: best_score=0.156458
fold 6: best_score=0.188367
fold 7: best_score=0.154549
fold 8: best_score=0.110516
[cv] mean=0.140379 std=0.029502


fold 1: best_score=0.168211
fold 2: best_score=0.106193
fold 3: best_score=0.099200
fold 4: best_score=0.136880
fold 5: best_score=0.155239
fold 6: best_score=0.187601
fold 7: best_score=0.154510
fold 8: best_score=0.110191
[cv] mean=0.139753 std=0.030021

In [24]:
# -----------------------------
# Fit final model on all 80 training genes, then write submission
# -----------------------------
def fit_full_model():
    model = BilinearDeltaModel(d_pert=EMB_DIM_PERT, d_out=EMB_DIM_OUT, rank_r=RANK_R, dropout=DROPOUT).to(device)
    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    best_score = -1e18
    best_state = None
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)
            loss = weighted_l1_like(Yt.index_select(0, b_t), pred)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t).detach().cpu().numpy().astype(np.float32)

            s = score_delta(Y, pred_np)
            sc = s["score"]
            print(f"epoch={epoch:4d} train_score={sc:.6f} wcos={s['wcos']:.6f} pred_wmae={s['pred_wmae']:.6f}")

            if sc > best_score:
                best_score = sc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model

model_final = fit_full_model()

def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    with torch.no_grad():
        y = model_final(z, Uo_t).detach().cpu().numpy().astype(np.float32)[0]
    return y

# Build submission from sample_submission.csv
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

# ensure order matches gene_columns
idx = {g: i for i, g in enumerate(gene_columns)}
perm = [idx[g] for g in sub_gene_cols]

# fill default baseline for unknown test perts
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

# fill known val perts (pert_1..pert_60)
hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

print(f"[ok] filled {hit} leaderboard rows from mapping")

out_path = ROOT / "submission_bilinear.csv"
sub.to_csv(out_path, index=False)
print("[ok] wrote:", out_path)


epoch=  25 train_score=0.199632 wcos=0.488504 pred_wmae=0.071151
epoch=  50 train_score=0.223055 wcos=0.513068 pred_wmae=0.069941
epoch=  75 train_score=0.262689 wcos=0.548469 pred_wmae=0.067838
epoch= 100 train_score=0.313922 wcos=0.588751 pred_wmae=0.065443
epoch= 125 train_score=0.368121 wcos=0.626206 pred_wmae=0.062969
epoch= 150 train_score=0.419295 wcos=0.656955 pred_wmae=0.060702
epoch= 175 train_score=0.464376 wcos=0.679179 pred_wmae=0.058745
epoch= 200 train_score=0.501616 wcos=0.694447 pred_wmae=0.057165
epoch= 225 train_score=0.533915 wcos=0.707576 pred_wmae=0.055904
epoch= 250 train_score=0.560998 wcos=0.716873 pred_wmae=0.054812
epoch= 275 train_score=0.585925 wcos=0.726104 pred_wmae=0.053923
epoch= 300 train_score=0.607713 wcos=0.732753 pred_wmae=0.053096
epoch= 325 train_score=0.627641 wcos=0.739853 pred_wmae=0.052429
epoch= 350 train_score=0.646601 wcos=0.746193 pred_wmae=0.051789
epoch= 375 train_score=0.660349 wcos=0.750544 pred_wmae=0.051312
epoch= 400 train_score=0.

epoch=  25 train_score=0.198746 wcos=0.487608 pred_wmae=0.071188
epoch=  50 train_score=0.219706 wcos=0.509443 pred_wmae=0.070065
epoch=  75 train_score=0.259733 wcos=0.545250 pred_wmae=0.067865
epoch= 100 train_score=0.311645 wcos=0.585495 pred_wmae=0.065357
epoch= 125 train_score=0.366496 wcos=0.624548 pred_wmae=0.062860
epoch= 150 train_score=0.417624 wcos=0.655439 pred_wmae=0.060559
epoch= 175 train_score=0.463975 wcos=0.678903 pred_wmae=0.058675
epoch= 200 train_score=0.502267 wcos=0.695064 pred_wmae=0.057094
epoch= 225 train_score=0.532521 wcos=0.706480 pred_wmae=0.055899
epoch= 250 train_score=0.558642 wcos=0.715405 pred_wmae=0.054898
epoch= 275 train_score=0.583674 wcos=0.724481 pred_wmae=0.053999
epoch= 300 train_score=0.603153 wcos=0.731410 pred_wmae=0.053319
epoch= 325 train_score=0.621887 wcos=0.737184 pred_wmae=0.052634
epoch= 350 train_score=0.639025 wcos=0.742880 pred_wmae=0.052035
epoch= 375 train_score=0.657187 wcos=0.748590 pred_wmae=0.051425
epoch= 400 train_score=0.670351 wcos=0.751868 pred_wmae=0.050928
epoch= 425 train_score=0.685498 wcos=0.756807 pred_wmae=0.050430
epoch= 450 train_score=0.697236 wcos=0.760652 pred_wmae=0.050028
epoch= 475 train_score=0.710665 wcos=0.764852 pred_wmae=0.049602
epoch= 500 train_score=0.722773 wcos=0.768706 pred_wmae=0.049198
epoch= 525 train_score=0.733332 wcos=0.771857 pred_wmae=0.048860
epoch= 550 train_score=0.744652 wcos=0.774799 pred_wmae=0.048471
epoch= 575 train_score=0.753307 wcos=0.777587 pred_wmae=0.048182
epoch= 600 train_score=0.763991 wcos=0.780026 pred_wmae=0.047843
epoch= 625 train_score=0.771716 wcos=0.782409 pred_wmae=0.047610
epoch= 650 train_score=0.781038 wcos=0.785228 pred_wmae=0.047304
epoch= 675 train_score=0.790654 wcos=0.787587 pred_wmae=0.047015
epoch= 700 train_score=0.796898 wcos=0.789635 pred_wmae=0.046827
epoch= 725 train_score=0.803267 wcos=0.790969 pred_wmae=0.046629
epoch= 750 train_score=0.810749 wcos=0.792662 pred_wmae=0.046363
epoch= 775 train_score=0.818036 wcos=0.795020 pred_wmae=0.046175
epoch= 800 train_score=0.824058 wcos=0.797259 pred_wmae=0.046006
epoch= 825 train_score=0.830690 wcos=0.798836 pred_wmae=0.045806
epoch= 850 train_score=0.837818 wcos=0.800406 pred_wmae=0.045591
epoch= 875 train_score=0.843421 wcos=0.802127 pred_wmae=0.045433
epoch= 900 train_score=0.848761 wcos=0.803281 pred_wmae=0.045261
epoch= 925 train_score=0.853170 wcos=0.803704 pred_wmae=0.045102
epoch= 950 train_score=0.858074 wcos=0.805542 pred_wmae=0.044974
epoch= 975 train_score=0.865319 wcos=0.806507 pred_wmae=0.044745
epoch=1000 train_score=0.866940 wcos=0.807401 pred_wmae=0.044703
[ok] filled 60 leaderboard rows from mapping
[ok] wrote: submission_bilinear.csv